In [4]:
import pandas as pd
from sqlalchemy import text
%run db_config.ipynb

def load_symbol_id_map(conn) -> dict[str, int]:
    rows = conn.execute(text("SELECT id, symbol FROM stocks")).fetchall()
    return {row.symbol: row.id for row in rows}

import yfinance as yf

def download_ohlc(symbol: str, period: str = "5y") -> pd.DataFrame:
    # yfinance uses hyphen for class shares: BRK.B -> BRK-B
    yf_symbol = symbol.replace(".", "-")
    df = yf.download(yf_symbol, period=period, interval="1d", auto_adjust=False)
    if df.empty:
        return df

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df = df.rename(columns={
        "Date": "trade_date",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
    })
    df = df[["trade_date", "open", "high", "low", "close", "volume"]]
    df["trade_date"] = pd.to_datetime(df["trade_date"]).dt.date
    df = df.dropna(subset=["open", "high", "low", "close", "volume"])
    return df

def insert_ohlc_for_symbol(conn, stock_id: int, df: pd.DataFrame) -> int:
    if df.empty:
        return 0

    df = df.copy()
    df["stock_id"] = stock_id
    records = df.to_dict(orient="records")

    conn.execute(
        text(
            "INSERT INTO stock_ohlc_data "
            "(trade_date, open, high, low, close, volume, stock_id) "
            "VALUES (:trade_date, :open, :high, :low, :close, :volume, :stock_id)"
        ),
        records,
    )
    return len(records)

import json
from pathlib import Path

CHECKPOINT_FILE = Path("python/checkpoint/ohlc_checkpoint.json")

def load_checkpoint() -> set[str]:
    if CHECKPOINT_FILE.exists():
        return set(json.loads(CHECKPOINT_FILE.read_text()))
    return set()

def save_checkpoint(done: set[str]) -> None:
    CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_FILE.write_text(json.dumps(sorted(done)))

def main():
    engine = get_engine()
    done = load_checkpoint()

    with engine.begin() as conn:
        symbol_map = load_symbol_id_map(conn)

    for symbol, stock_id in symbol_map.items():
        if symbol in done:
            print(f"SKIP {symbol} (checkpoint)")
            continue

        try:
            df = download_ohlc(symbol)
            if df.empty:
                print(f"WARN {symbol}: no data")
                done.add(symbol)
                save_checkpoint(done)
                continue

            with engine.begin() as conn:
                count = insert_ohlc_for_symbol(conn, stock_id, df)

            print(f"OK {symbol}: inserted {count} rows")
            done.add(symbol)
            save_checkpoint(done)

        except Exception as e:
            print(f"FAIL {symbol}: {e}")

    print(f"\nCompleted {len(done)} / {len(symbol_map)} symbols")

if __name__ == "__main__":
    main()

[*********************100%***********************]  1 of 1 completed


OK MMM: inserted 1255 rows


[*********************100%***********************]  1 of 1 completed


OK AOS: inserted 1255 rows


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

...

OK YUM: inserted 1255 rows



[*********************100%***********************]  1 of 1 completed

OK ZBRA: inserted 1255 rows



[*********************100%***********************]  1 of 1 completed

OK ZBH: inserted 1255 rows
OK ZTS: inserted 1255 rows

Completed 503 / 503 symbols
